In [ ]:
!pip install playwright
!playwright install chromium
!pip install playwright nest_asyncio
!playwright install chromium

In [ ]:
!apt-get update
!apt-get install -y \
    libatk1.0-0 \
    libatk-bridge2.0-0 \
    libcups2 \
    libxkbcommon0 \
    libxcomposite1 \
    libxdamage1 \
    libxrandr2 \
    libgbm1 \
    libasound2 \
    libpangocairo-1.0-0 \
    libpango-1.0-0 \
    libcairo2 \
    libatspi2.0-0 \
    libnss3 \
    libxss1

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import asyncio
import nest_asyncio
from playwright.async_api import async_playwright
from PIL import Image
import os

nest_asyncio.apply()

HTML_FILE = "instagram_carousel_mockup.html"
OUTPUT_PDF = "slides_high_quality.pdf"

# tweak these for quality
WIDTH = 2160     # 2x Instagram resolution
HEIGHT = 2160
SCALE = 2        # device pixel ratio boost

async def generate_pdf():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            args=["--no-sandbox", "--disable-setuid-sandbox"]
        )

        context = await browser.new_context(
            viewport={"width": WIDTH, "height": HEIGHT},
            device_scale_factor=SCALE   # key for sharpness
        )

        page = await context.new_page()

        await page.goto(f"file://{os.path.abspath(HTML_FILE)}")
        await page.wait_for_selector(".slide")

        total_slides = await page.eval_on_selector_all(
            ".slide", "els => els.length"
        )
        print(f"Found {total_slides} slides")

        image_paths = []

        for i in range(total_slides):
            await page.evaluate(f"""
                () => {{
                    const slides = document.querySelectorAll('.slide');
                    slides.forEach((s, idx) => {{
                        s.style.display = (idx === {i}) ? 'flex' : 'none';
                    }});
                }}
            """)

            await page.wait_for_timeout(300)

            img_path = f"slide_{i}.png"

            # HIGH-QUALITY SCREENSHOT
            await page.locator(".frame").screenshot(
                path=img_path,
                type="png"   # PNG = no compression loss
            )

            image_paths.append(img_path)

        await browser.close()

    # Convert to PDF WITHOUT resizing
    images = []
    for pth in image_paths:
        img = Image.open(pth).convert("RGB")
        images.append(img)

    images[0].save(
        OUTPUT_PDF,
        save_all=True,
        append_images=images[1:],
        resolution=300.0   # better print density
    )

    print("Saved:", OUTPUT_PDF)


await generate_pdf()

In [ ]:
from google.colab import files
files.download("slides.pdf")